In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

In [12]:
os.chdir('/content/drive/MyDrive/ZoomCamp')

os.getcwd()

'/content/drive/MyDrive/ZoomCamp'

In [13]:
!git clone "https://github.com/DataTalksClub/machine-learning-zoomcamp.git"

Cloning into 'machine-learning-zoomcamp'...
remote: Enumerating objects: 18030, done.
remote: Counting objects: 100% (951/951), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 18030 (delta 917), reused 901 (delta 901), pack-reused 17079 (from 3)
Receiving objects: 100% (18030/18030), 996.11 MiB | 12.40 MiB/s, done.
Resolving deltas: 100% (11276/11276), done.
Updating files: 100% (1785/1785), done.


In [17]:
os.chdir("machine-learning-zoomcamp/cohorts/2026/homework/05-deployment")
os.getcwd()


'/content/drive/MyDrive/ZoomCamp/machine-learning-zoomcamp/cohorts/2026/homework/05-deployment'

In [19]:

!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [20]:
!uv sync --locked

Using CPython 3.11.15
Creating virtual environment at: .venv
Resolved 19 packages in 2ms
Prepared 18 packages in 2.00s
░░░░░░░░░░░░░░░░░░░░ [0/18] Installing wheels...                                warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 18 packages in 1m 28s
 + annotated-types==0.8.0
 + anyio==4.15.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + fastapi==0.119.1
 + h11==0.16.0
 + idna==3.19
 + joblib==1.6.0
 + numpy==2.3.3
 + pydantic==2.13.5
 + pydantic-core==2.46.5
 + scikit-learn==1.7.2
 + scipy==1.17.1
 + starlette==0.48.0
 + threadpoolctl==3.6.0
 + typing-extensions==4.16.0
 + typing-inspection==0.4.4
 + uvicorn==0.38.0


In [53]:

!sha256sum pipeline.bin

!uv run python smoke_test.py

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin
{'conversion_probability': 0.769799, 'conversion': True}


In [40]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [41]:
!uv run python -c "import sklearn; print(sklearn.__version__)"

1.7.2


In [42]:
import pickle

with open('pipeline.bin', 'rb') as f:
  dv, model = pickle.load(f)

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or in

In [43]:
data = {
  "lead_source": "paid_ads",
  "industry": "technology",
  "employment_status": "employed",
  "location": "north_america",
  "number_of_courses_viewed": 2,
  "annual_income": 79276.0,
  "interaction_count": 4,
  "lead_score": 0.41
}

In [44]:
X = dv.transform([data])
y_pred = model.predict_proba(X)[0, 1]
convert = y_pred >= 0.5
result = {
      'conversion_probability': round(float(y_pred), 3),
      'convert': bool(churn)
  }

result

{'conversion_probability': 0.533, 'convert': True}

In [46]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696

INFO:     Started server process [19736]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [19736]


In [48]:
!source .venv/bin/activate

In [51]:
!cat Dockerfile -n 2

     1	FROM python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
     2	
     3	COPY --from=ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4 /uv /uvx /bin/
     4	
     5	ENV PATH="/code/.venv/bin:$PATH" \
     6	    UV_COMPILE_BYTECODE=1 \
     7	    UV_LINK_MODE=copy
     8	
     9	WORKDIR /code
    10	COPY pyproject.toml uv.lock .python-version ./
    11	RUN uv sync --locked --no-dev
    12	
    13	COPY model.py predict.py feature_defaults.json model_metadata.json pipeline.bin q6_test.py ./
    14	
    15	EXPOSE 9696
    16	ENTRYPOINT ["uvicorn", "predict:app", "--host", "0.0.0.0", "--port", "9696"]
cat: 2: No such file or directory


In [54]:
!uv run python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [55]:
!curl -s http://localhost:9696/health

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}